In [1]:
import pandas as pd

main = pd.read_csv("../data/pakwheels_used_cars_2026_clean (2).csv")
other = pd.read_csv("../data/Pakistan Used Car Prices (2).csv")

print(main.shape)
print(other.shape)

(10589, 17)
(59554, 9)


In [2]:
# 1) Dono files mein model ka naam same format mein karo (chhote letters, extra space nahi)
main["brand_k"]  = main["brand"].str.strip().str.lower()
main["model_k"]  = main["model"].str.strip().str.lower()
other["brand_k"] = other["brand"].str.strip().str.lower()
other["model_k"] = other["model_name"].str.strip().str.lower()

# 2) Doosri file se har (brand, model) ka summary banao:
#    kitni gaariyan hain, aur average kitni chali hain
market = (other.groupby(["brand_k", "model_k"])
          .agg(market_supply_count=("Km_Driven", "size"),
               market_avg_mileage=("Km_Driven", "mean"))
          .reset_index())

# 3) Ye summary pehli file mein jod do
df = main.merge(market, on=["brand_k", "model_k"], how="left")

print(df.shape)
print(df[["brand", "model", "market_supply_count", "market_avg_mileage"]].head())

(10589, 21)
     brand   model  market_supply_count  market_avg_mileage
0   Toyota   Prado                825.0        91225.169697
1  Changan  Alsvin                254.0        32247.122047
2   Suzuki  Cultus               2653.0        80242.158311
3    Honda    City               2116.0       110851.370038
4  Hyundai   Santa                  NaN                 NaN


In [3]:
print(market.sort_values(by="market_supply_count", ascending=False).head(15))

      brand_k              model_k  market_supply_count  market_avg_mileage
457    toyota              corolla                 7584       107019.899657
142     honda                civic                 3959        97029.769891
415    suzuki               mehran                 3068        87774.063885
391    suzuki                 alto                 2894        70956.784727
401    suzuki               cultus                 2653        80242.158311
139     honda                 city                 2116       110851.370038
430    suzuki              wagon r                 1646        65631.280073
140     honda  city 5th generation                 1445        92059.080277
524    toyota                 vitz                 1430       102559.069231
85   daihatsu                 mira                 1124        59028.703737
175     honda                vezel                 1080        82629.431481
427    suzuki                swift                  941        74986.116897
500    toyot

In [4]:
# Ab Brand, Model AUR Year teeno ke hisaab se group karein
market_by_year = (other.groupby(["brand_k", "model_k", "year"])
                  .agg(market_supply_count=("Km_Driven", "size"),
                       market_avg_mileage=("Km_Driven", "mean"))
                  .reset_index())

# Sirf Toyota Corolla ke 2015 se 2024 tak ke models dekhein
print(market_by_year[(market_by_year["brand_k"] == "toyota") & 
                     (market_by_year["model_k"] == "corolla") & 
                     (market_by_year["year"] >= 2015)])

     brand_k  model_k  year  market_supply_count  market_avg_mileage
2608  toyota  corolla  2015                  539       115072.682746
2609  toyota  corolla  2016                  501       108779.754491
2610  toyota  corolla  2017                  574        96684.290941
2611  toyota  corolla  2018                  644        79263.838509
2612  toyota  corolla  2019                  543        72520.012891
2613  toyota  corolla  2020                  341        62915.539589
2614  toyota  corolla  2021                  398        40262.944724
2615  toyota  corolla  2022                  507        23599.988166
2616  toyota  corolla  2023                  139        10700.460432
2617  toyota  corolla  2024                   41          973.634146


In [ ]:
# 1) 2024 tak asli saal rahega, sirf 2025/2026 match hone ke liye 2024 use karenge
main["year_k"]  = main["year"].clip(upper=2024)
other["year_k"] = other["year"]

# 2) Ab Brand, Model AUR Year teeno par group karein
market = (other.groupby(["brand_k", "model_k", "year_k"])
          .agg(market_supply_count=("Km_Driven", "size"),
               market_avg_mileage=("Km_Driven", "mean"))
          .reset_index())

# 3) Teeno keys (brand_k, model_k, year_k) par merge karein
df = main.merge(market, on=["brand_k", "model_k", "year_k"], how="left")

# Check 
print(df[["brand", "model", "year", "market_supply_count", "market_avg_mileage"]].head(10))

     brand     model  year  market_supply_count  market_avg_mileage
0   Toyota     Prado  2003                 21.0       132487.428571
1  Changan    Alsvin  2022                105.0        28038.980952
2   Suzuki    Cultus  2022                 97.0        24989.020619
3    Honda      City  2025                 14.0         8192.928571
4  Hyundai     Santa  2026                  NaN                 NaN
5  Hyundai    Santro  2004                 64.0       117104.484375
6   Toyota     Prius  2011                161.0       144761.813665
7      KIA  Sportage  2021                336.0        44130.401786
8   Suzuki    Cultus  2019                148.0        75253.689189
9    Honda     Civic  2021                384.0        38454.609375


In [9]:
main["year_k"] = main["year"].clip(upper=2024)
print(main[["year","year_k"]].head(5))

   year  year_k
0  2003    2003
1  2022    2022
2  2022    2022
3  2025    2024
4  2026    2024


In [11]:
df["vehicle_age"] = 2026 - df["year"]
print(df[["brand", "model", "year", "vehicle_age"]].head(20))

      brand     model  year  vehicle_age
0    Toyota     Prado  2003           23
1   Changan    Alsvin  2022            4
2    Suzuki    Cultus  2022            4
3     Honda      City  2025            1
4   Hyundai     Santa  2026            0
5   Hyundai    Santro  2004           22
6    Toyota     Prius  2011           15
7       KIA  Sportage  2021            5
8    Suzuki    Cultus  2019            7
9     Honda     Civic  2021            5
10   Toyota   Corolla  2021            5
11    Honda     Civic  2014           12
12    Honda      City  2017            9
13   Toyota   Corolla  2006           20
14   Suzuki    Cultus  2004           22
15   Suzuki    Khyber  1990           36
16   Toyota     Yaris  2021            5
17   Toyota      Aqua  2015           11
18    Honda     Civic  2015           11
19    Haval        H6  2025            1


In [15]:
cols_9 = [
    "brand", "model", "variant", "year", "mileage_km", 
    "fuel_type", "transmission", "engine_cc", "city"
]
print(main[cols_9].head())

     brand   model                               variant  year  mileage_km  \
0   Toyota   Prado          3rd (J120) Generation TZ 3.4  2003    161969.0   
1  Changan  Alsvin                       1.3L MT Comfort  2022     59216.0   
2   Suzuki  Cultus                                   VXL  2022     22120.0   
3    Honda    City  6th (GM4) Generation 1.5L ASPIRE CVT  2025     14339.0   
4  Hyundai   Santa                          Fe Signature  2026        10.0   

  fuel_type transmission  engine_cc        city  
0    Petrol    Automatic     3400.0      Lahore  
1    Petrol       Manual     1300.0      Lahore  
2    Petrol       Manual     1000.0     Karachi  
3    Petrol    Automatic     1500.0  Rawalpindi  
4    Hybrid    Automatic     1600.0     Karachi  


In [17]:
cols_11 = [
    "brand", "model", "variant", "year", "mileage_km", "fuel_type", "transmission", "engine_cc", "city",
    "market_supply_count", "market_avg_mileage"
]
print(df[cols_11].head())

     brand   model                               variant  year  mileage_km  \
0   Toyota   Prado          3rd (J120) Generation TZ 3.4  2003    161969.0   
1  Changan  Alsvin                       1.3L MT Comfort  2022     59216.0   
2   Suzuki  Cultus                                   VXL  2022     22120.0   
3    Honda    City  6th (GM4) Generation 1.5L ASPIRE CVT  2025     14339.0   
4  Hyundai   Santa                          Fe Signature  2026        10.0   

  fuel_type transmission  engine_cc        city  market_supply_count  \
0    Petrol    Automatic     3400.0      Lahore                 21.0   
1    Petrol       Manual     1300.0      Lahore                105.0   
2    Petrol       Manual     1000.0     Karachi                 97.0   
3    Petrol    Automatic     1500.0  Rawalpindi                 14.0   
4    Hybrid    Automatic     1600.0     Karachi                  NaN   

   market_avg_mileage  
0       132487.428571  
1        28038.980952  
2        24989.020619  
3 

In [18]:
FINAL_TABLE = df.copy()

In [19]:
FINAL_TABLE[cols_11].head()

,brand,model,variant,year,mileage_km,fuel_type,transmission,engine_cc,city,market_supply_count,market_avg_mileage
0,Toyota,Prado,3rd (J120) Generation TZ 3.4,2003,161969.0,Petrol,Automatic,3400.0,Lahore,21.0,132487.428571
1,Changan,Alsvin,1.3L MT Comfort,2022,59216.0,Petrol,Manual,1300.0,Lahore,105.0,28038.980952
2,Suzuki,Cultus,VXL,2022,22120.0,Petrol,Manual,1000.0,Karachi,97.0,24989.020619
3,Honda,City,6th (GM4) Generation 1.5L ASPIRE CVT,2025,14339.0,Petrol,Automatic,1500.0,Rawalpindi,14.0,8192.928571
4,Hyundai,Santa,Fe Signature,2026,10.0,Hybrid,Automatic,1600.0,Karachi,NaN,NaN


In [20]:
FINAL_TABLE["market_avg_mileage"] = FINAL_TABLE["market_avg_mileage"].round(0)

In [21]:
FINAL_TABLE[cols_11].head()

,brand,model,variant,year,mileage_km,fuel_type,transmission,engine_cc,city,market_supply_count,market_avg_mileage
0,Toyota,Prado,3rd (J120) Generation TZ 3.4,2003,161969.0,Petrol,Automatic,3400.0,Lahore,21.0,132487.0
1,Changan,Alsvin,1.3L MT Comfort,2022,59216.0,Petrol,Manual,1300.0,Lahore,105.0,28039.0
2,Suzuki,Cultus,VXL,2022,22120.0,Petrol,Manual,1000.0,Karachi,97.0,24989.0
3,Honda,City,6th (GM4) Generation 1.5L ASPIRE CVT,2025,14339.0,Petrol,Automatic,1500.0,Rawalpindi,14.0,8193.0
4,Hyundai,Santa,Fe Signature,2026,10.0,Hybrid,Automatic,1600.0,Karachi,NaN,NaN


In [23]:
FINAL_TABLE["vehicle_age"] = 2026 - FINAL_TABLE["year"]


In [24]:
FINAL_TABLE["km_per_year"] = (FINAL_TABLE["mileage_km"] / FINAL_TABLE["vehicle_age"].clip(lower=1))

In [25]:
FINAL_TABLE["km_per_year"] = FINAL_TABLE["km_per_year"].round(0)

In [26]:
cols_13 = [
    "brand", "model", "variant", "year", "mileage_km", 
    "fuel_type", "transmission", "engine_cc", "city","market_supply_count",
    "market_avg_mileage","vehicle_age","km_per_year"
]
FINAL_TABLE[cols_13].head()

,brand,model,variant,year,mileage_km,fuel_type,transmission,engine_cc,city,market_supply_count,market_avg_mileage,vehicle_age,km_per_year
0,Toyota,Prado,3rd (J120) Generation TZ 3.4,2003,161969.0,Petrol,Automatic,3400.0,Lahore,21.0,132487.0,23,7042.0
1,Changan,Alsvin,1.3L MT Comfort,2022,59216.0,Petrol,Manual,1300.0,Lahore,105.0,28039.0,4,14804.0
2,Suzuki,Cultus,VXL,2022,22120.0,Petrol,Manual,1000.0,Karachi,97.0,24989.0,4,5530.0
3,Honda,City,6th (GM4) Generation 1.5L ASPIRE CVT,2025,14339.0,Petrol,Automatic,1500.0,Rawalpindi,14.0,8193.0,1,14339.0
4,Hyundai,Santa,Fe Signature,2026,10.0,Hybrid,Automatic,1600.0,Karachi,NaN,NaN,0,10.0


In [ ]:
FINAL_TABLE["is_hybrid_or_ev"] = FINAL_TABLE["fuel_type"].isin(["Hybrid", "Electric", "PHEV"]).astype(int)


In [37]:
FINAL_TABLE["city"].value_counts().head(5)

city
Lahore        2991
Karachi       2131
Islamabad     1894
Rawalpindi     754
Faisalabad     389
Name: count, dtype: int64

In [38]:
print(list(FINAL_TABLE["city"].unique()))

['Lahore', 'Karachi', 'Rawalpindi', 'Islamabad', 'Multan', 'Mandi Bahauddin', 'Peshawar', 'Gujranwala', 'Faisalabad', 'Mian Wali', 'Rahim Yar Khan', 'Hazro', 'Bhakkar', 'Sumandari', 'Mirpur Ak', 'Mardan', 'Chakwal', 'Wah Cantt', 'Muzaffar Gargh', 'Sialkot', 'Nankana Sahib', 'Mehrabpur', 'Abbottabad', 'Gaggo Mandi', 'Bahawalpur', 'Jhelum', 'Gujrat', 'Nawabshah', 'Sahiwal', 'Layyah', 'Sheikhupura', 'Hafizabad', 'Sargodha', 'Hyderabad', 'Gojra', 'Mansehra', 'Pindi Gheb', 'Haripur', 'Jauharabad', 'Dgkhan', 'Swabi', 'Tarbela', 'Quetta', 'Rawat', 'Talagang', 'Kohat', 'Muridke', 'Okara', 'Bahawalnagar', 'Buner', 'Chichawatni', 'Charsadda', 'Kallar Saddiyian', 'Sadiqabad', 'Dina', 'Phalia', 'Kharian', 'Lodhran', 'Burewala', 'Sukkur', 'Shahdadpur', 'Jhang', 'Jand', 'Kasur', 'Dadyal Ak', 'Jacobabad', 'Topi', 'Khushab', 'Swat', 'Arifwala', 'Pir Mahal', 'Farooqabad', 'Khanewal', 'Attock', 'Jalalpur Pirwala', 'Jahaniya', 'Daska', 'Nowshera', 'Kot Addu', 'Narowal', 'Wazirabad', 'Kamoke', 'Muzaffarab

In [39]:
Sindh_list = [
    # CSV wale 23 shehar:
    "Karachi", "Mehrabpur", "Nawabshah", "Hyderabad", "Sukkur", "Shahdadpur", 
    "Jacobabad", "Ghotki", "Mirpur Mathelo", "Moro", "Larkana", "Noudero", 
    "Mirpur Khas", "Badin", "Khairpur Mir", "Tando Adam", "Dadu", "Daulatpur", 
    "Pano Aqil", "Sanghar", "Daharki", "Makli", "Tando Muhammad Khan",
    # Extra major Sindh cities:
    "Thatta", "Shikarpur", "Umerkot", "Kotri", "Jamshoro", "Tando Allahyar", 
    "Matiari", "Kandhkot", "Kashmore", "Tharparkar", "Mithi", "Sujawal", "Hala", "Sehwan"
]

Kpk_list = [
    # CSV wale 26 shehar:
    "Peshawar", "Mardan", "Abbottabad", "Mansehra", "Haripur", "Swabi", 
    "Tarbela", "Kohat", "Buner", "Charsadda", "Topi", "Swat", "Nowshera", 
    "Takhtbai", "Ghazi", "Dera Ismail Khan", "Jehangira", "Risalpur", 
    "Swatmingora", "Mingora", "Bannu", "Nowshera Cantt", "Qalandarabad", 
    "Akora Khattak", "Saidu Sharif", "Malakand Agency",
    # Extra major KPK cities:
    "Chitral", "Dir", "Timergara", "Karak", "Hangu", "Tank", "Lakki Marwat", 
    "Batkhela", "Shabqadar", "Parachinar", "Besham", "Balakot", "Naran"
]

Balochistan_list = [
    "Quetta", "Gwadar", "Hub", "Khuzdar", "Turbat", "Chaman", "Sibi", 
    "Zhob", "Loralai", "Pishin", "Kalat", "Mastung", "Pasni", "Ormara", 
    "Dera Murad Jamali", "Usta Mohammad", "Lasbela", "Ziarat", "Nushki"
]

Azad_Jammu_Kashmir_list = [
    "Mirpur Ak", "Dadyal Ak", "Muzaffarabad", "Mangla", "Bhimber", "Kashmir", "Hattian",
    "Rawalakot", "Kotli", "Bagh", "Neelum", "Sudhanoti", "Haveli", "Gilgit", "Skardu", "Hunza"
]

def get_province(city):
    if city == "Islamabad":
        return "Islamabad"
    elif city in Sindh_list:
        return "Sindh"
    elif city in Kpk_list:
        return "KPK"
    elif city in Balochistan_list:
        return "Balochistan"
    elif city in Azad_Jammu_Kashmir_list:
        return "AJK"
    else:
        return "Punjab"

FINAL_TABLE["province"] = FINAL_TABLE["city"].apply(get_province)
FINAL_TABLE["province"].value_counts()

province
Punjab         5816
Sindh          2244
Islamabad      1894
KPK             570
Balochistan      37
AJK              28
Name: count, dtype: int64

In [40]:
cols_15 = [
    "brand", "model", "variant", "year", "mileage_km", 
    "fuel_type", "transmission", "engine_cc", "city","market_supply_count",
    "market_avg_mileage","vehicle_age","km_per_year","is_hybrid_or_ev","province"
]
FINAL_TABLE[cols_15].head()

,brand,model,variant,year,mileage_km,fuel_type,transmission,engine_cc,city,market_supply_count,market_avg_mileage,vehicle_age,km_per_year,is_hybrid_or_ev,province
0,Toyota,Prado,3rd (J120) Generation TZ 3.4,2003,161969.0,Petrol,Automatic,3400.0,Lahore,21.0,132487.0,23,7042.0,0,Punjab
1,Changan,Alsvin,1.3L MT Comfort,2022,59216.0,Petrol,Manual,1300.0,Lahore,105.0,28039.0,4,14804.0,0,Punjab
2,Suzuki,Cultus,VXL,2022,22120.0,Petrol,Manual,1000.0,Karachi,97.0,24989.0,4,5530.0,0,Sindh
3,Honda,City,6th (GM4) Generation 1.5L ASPIRE CVT,2025,14339.0,Petrol,Automatic,1500.0,Rawalpindi,14.0,8193.0,1,14339.0,0,Punjab
4,Hyundai,Santa,Fe Signature,2026,10.0,Hybrid,Automatic,1600.0,Karachi,NaN,NaN,0,10.0,1,Sindh


In [41]:
import numpy as np

local_models = [
    # 1. Pak Suzuki (New + Old Used Local)
    "alto", "wagon r", "cultus", "swift", "mehran", "bolan", "ravi", 
    "baleno", "liana", "khyber", "margalla", "potohar", "ciaz",
    
    # 2. Toyota Indus & Daihatsu (New + Old Used Local)
    "corolla", "yaris", "corolla cross", "fortuner", "hilux", "revo", "cuore",
    
    # 3. Honda Atlas (New + Old Used Local)
    "city", "civic", "hr-v", "br-v",
    
    # 4. Hyundai Nishat (New + Old Used Local)
    "tucson", "elantra", "sonata", "santa", "santa fe", "porter", "santro", "shehzore",
    
    # 5. KIA Lucky Motors & Peugeot
    "picanto", "sportage", "sorento", "stonic", "2008", "grand carnival",
    
    # 6. Changan, Haval, MG, Chery, Proton, DFSK, Prince, United, Isuzu
    "alsvin", "oshan", "karvaan", "sherpa",
    "h6", "jolion", "hs", "zs",
    "tiggo 4 pro", "tiggo 8 pro", "tiggo",
    "saga", "x70", "glory", "glory 580",
    "pearl", "bravo", "alpha", "d-max", "sunny", "bj40", "j7", "j6"
]

FINAL_TABLE["assembly"] = np.where(FINAL_TABLE["model_k"].isin(local_models), "Local", "Imported")
FINAL_TABLE["assembly"].value_counts()

assembly
Local       7500
Imported    3089
Name: count, dtype: int64

In [42]:
cols_15 = [
    "brand", "model", "variant", "year", "mileage_km", 
    "fuel_type", "transmission", "engine_cc", "city",
    "market_supply_count", "market_avg_mileage",
    "vehicle_age", "km_per_year", "province", "assembly"
]

FINAL_TABLE[cols_15].head()

,brand,model,variant,year,mileage_km,fuel_type,transmission,engine_cc,city,market_supply_count,market_avg_mileage,vehicle_age,km_per_year,province,assembly
0,Toyota,Prado,3rd (J120) Generation TZ 3.4,2003,161969.0,Petrol,Automatic,3400.0,Lahore,21.0,132487.0,23,7042.0,Punjab,Imported
1,Changan,Alsvin,1.3L MT Comfort,2022,59216.0,Petrol,Manual,1300.0,Lahore,105.0,28039.0,4,14804.0,Punjab,Local
2,Suzuki,Cultus,VXL,2022,22120.0,Petrol,Manual,1000.0,Karachi,97.0,24989.0,4,5530.0,Sindh,Local
3,Honda,City,6th (GM4) Generation 1.5L ASPIRE CVT,2025,14339.0,Petrol,Automatic,1500.0,Rawalpindi,14.0,8193.0,1,14339.0,Punjab,Local
4,Hyundai,Santa,Fe Signature,2026,10.0,Hybrid,Automatic,1600.0,Karachi,NaN,NaN,0,10.0,Sindh,Local


In [43]:
FINAL_TABLE["price_in_pkr"] = FINAL_TABLE["price_pkr"]

In [44]:
final_16_cols = cols_15 + ["price_in_pkr"]
FINAL_TABLE[final_16_cols].to_csv("../data/kaarwala_merged_dataset.csv", index=False)